# SegmentEveryForam in Google Colab

Select **Runtime → Change runtime type → T4 GPU** before running anything. The bottom-right corner of the notebook should show **T4**, not just **Python 3**.

Run the installation cell first. When it finishes, Colab restarts the session once so the browser editor can load. Colab may say the session **crashed** or **restarted**: this is expected. Continue with **Check the browser editor**, and do not rerun the installation cell.

Upload one microscope image, run automatic box prompting, and review the numbered masks. Use the optional browser editor to correct masks or draw the scale bar before exporting.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "This session has no GPU. Select Runtime → Change runtime type → T4 GPU, "
        "then run this cell again."
    )

%pip -q install rtree rasterio ipympl
%env SAM2_BUILD_CUDA=0
%pip -q install --no-build-isolation "git+https://github.com/facebookresearch/sam2.git"
%pip -q install --no-deps "git+https://github.com/Geogabriel/SegmentEveryForam.git"

# Matplotlib only finds the newly installed browser editor (ipympl) after a
# restart, so restart the session once now.
print("Installation finished. Colab will now restart the session once.")
print("If Colab says the session crashed or restarted, that is expected.")
print("Continue with 'Check the browser editor'. Do not rerun this cell.")
_ = get_ipython().kernel.do_shutdown(restart=True)

### Check the browser editor

Run this after the installation cell has restarted the session.

In [ ]:
from google.colab import output
import matplotlib.pyplot as plt

browser_editor_available = False
output.enable_custom_widget_manager()
try:
    plt.switch_backend("module://ipympl.backend_nbagg")
except (ImportError, ValueError):
    print("The browser editor is not available yet.")
    print("Select Runtime → Restart session, then run this cell again. Do not rerun the installation cell.")
    print("You can also skip the editor and use numbered exclusions and an existing pixels-per-meter calibration.")
else:
    browser_editor_available = True
    print("The interactive Matplotlib backend is enabled.")

### Load the models

Colab downloads the fine-tuned U-Net checkpoint from GitHub and SAM 2.1 Large from Hugging Face. Both files are cached for the current session.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory, mkdtemp
from urllib.request import urlretrieve
from zipfile import is_zipfile

import matplotlib.pyplot as plt
import numpy as np
import torch
from google.colab import files
from sam2.build_sam import build_sam2

import segmenteveryforam as sef
from segmenteveryforam.specimen_export import export_specimens

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before loading the models.")

model_dir = Path("/content/sef_models")
model_dir.mkdir(exist_ok=True)
unet_path = model_dir / "best_val_loss.keras"
checkpoint_url = "https://raw.githubusercontent.com/Geogabriel/SegmentEveryForam/1f311172938d2265de0b2d54f05d1244f619057b/models/best_val_loss.keras"

if not is_zipfile(unet_path):
    with TemporaryDirectory(dir=model_dir) as temp_dir:
        pending = Path(temp_dir) / unet_path.name
        urlretrieve(checkpoint_url, pending)
        if not is_zipfile(pending):
            raise ValueError("GitHub did not return a valid .keras archive.")
        pending.replace(unet_path)

unet = sef.load_unet(unet_path)
sam_path = model_dir / "sam2.1_hiera_large.pt"
if not sam_path.is_file():
    with TemporaryDirectory(dir=model_dir) as temp_dir:
        pending = Path(temp_dir) / sam_path.name
        urlretrieve(
            "https://huggingface.co/facebook/sam2.1-hiera-large/resolve/main/sam2.1_hiera_large.pt",
            pending,
        )
        pending.replace(sam_path)
sam = build_sam2("configs/sam2.1/sam2.1_hiera_l.yaml", str(sam_path), device="cuda")

### Upload and segment one image

In [ ]:
editor = None

image_dir = Path(mkdtemp(prefix="sef_image_", dir="/content"))
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one microscope image.")
filename, image_bytes = next(iter(uploaded.items()))
image_path = image_dir / Path(filename).name
image_path.write_bytes(image_bytes)

image = sef.load_image(str(image_path))
polygons, _, boxes = sef.predict_with_box_prompts(image, unet, sam)
detected_grains = sef.polygons_to_grains(polygons, image=image)

fig, ax = plt.subplots(figsize=(14, 10))
sef.plot_image_w_colorful_grains(
    image, [grain.polygon for grain in detected_grains], ax,
    cmap="tab20b", plot_image=True, im_alpha=1.0,
)
for review_id, grain in enumerate(detected_grains, start=1):
    centroid = grain.polygon.centroid
    ax.text(centroid.x, centroid.y, str(review_id), color="yellow", ha="center",
            bbox=dict(facecolor="black", alpha=0.6, pad=1, edgecolor="none"))
ax.set_title(f"Review {len(detected_grains)} detected specimens")
plt.show()

### Edit masks and draw the scale bar (optional)

Before running this cell, enter the length printed on your scale bar in `scale_bar_um` (for example, `500`).

- Click an existing mask to select it, then press **D** to delete it or **M** to merge selected masks.
- Click an unmasked specimen to add it with SAM.
- Hold **Shift** and drag from one end of the scale bar to the other. A label on the red line shows its length in pixels and, if `scale_bar_um` is set, the resulting calibration, for example `1012.4 px = 500 µm`.

If you left `scale_bar_um` empty, the label shows only the pixel length. Enter the printed length in the export cell instead. When finished, continue to the export cell.

In [ ]:
if not browser_editor_available:
    raise RuntimeError("Run Check the browser editor first, or skip editing and use an existing pixels-per-meter calibration.")

from sam2.sam2_image_predictor import SAM2ImagePredictor

scale_bar_um = None  # Printed scale-bar length in µm, for example 500. You can also enter it in the export cell.
if scale_bar_um is not None and (not np.isfinite(scale_bar_um) or scale_bar_um <= 0):
    raise ValueError("Enter a positive scale-bar length in micrometers.")

scale_bar_um_at_launch = scale_bar_um

if editor is not None:
    editor.deactivate()

predictor = SAM2ImagePredictor(sam)
predictor.set_image(image)
editor = sef.ForamPlot(
    grains=sef.polygons_to_grains([grain.polygon for grain in detected_grains], image=image),
    image=image,
    predictor=predictor,
    blit=False,
    scale_m=scale_bar_um * 1e-6 if scale_bar_um is not None else 1.0,
    px_per_m=1.0,
    figsize=(12, 8),
)
editor.activate()
plt.show()

### Export measurements, crops, and masks

Enter the sample information. If you opened the editor, export uses its current masks and closes editing. Otherwise, you can exclude numbers from the original review overlay.

If you entered `scale_bar_um` in the editor cell, the export uses it automatically and you can leave `scale_bar_um` here as `None`. Enter it here only if you skipped it in the editor.

Set `export_crops_and_masks = False` to export only the measurements table. Otherwise, the export also includes an image crop and a binary mask for each specimen, named with the same consecutive `foram_id` values as the table.

In [ ]:
sample_id = image_path.stem
species = ""
depth_ccsf = None
age_ma = None
exclude_review_ids = []  # Only when skipping the editor. Example: [3, 12]

export_crops_and_masks = True  # False exports only the measurements table.

pixels_per_meter = None  # Optional, if reusing a known calibration.
scale_bar_um = None  # Only if you did not enter it in the editor cell.

if pixels_per_meter is None:
    if editor is None or editor.scale_line is None:
        raise ValueError("Draw the scale bar in the editor or enter an existing pixels_per_meter value.")
    bar_pixels = editor.px_per_m * editor.scale_m
    if scale_bar_um is None:
        scale_bar_um = scale_bar_um_at_launch
    if scale_bar_um is None:
        raise ValueError(f"The drawn line is {bar_pixels:.2f} px. Enter its printed length in scale_bar_um above.")
    if not np.isfinite(scale_bar_um) or scale_bar_um <= 0:
        raise ValueError("Enter the positive scale-bar length in micrometers in scale_bar_um.")
    pixels_per_meter = bar_pixels / (scale_bar_um * 1e-6)
    print(f"Scale bar: {bar_pixels:.1f} px = {scale_bar_um:g} µm")
if not np.isfinite(pixels_per_meter) or pixels_per_meter <= 0:
    raise ValueError("Scale calibration must be positive and finite.")

print(f"Scale calibration: {pixels_per_meter * 1e-6:.4g} px per µm ({pixels_per_meter:,.2f} px per meter)")

excluded = set(exclude_review_ids)
if excluded.difference(range(1, len(detected_grains) + 1)):
    raise ValueError("An excluded number does not match the review overlay.")
if editor is not None:
    if excluded:
        raise ValueError("Remove unwanted masks in the editor or clear exclude_review_ids.")
    grains = editor.get_grains()
else:
    grains = [grain for i, grain in enumerate(detected_grains, start=1) if i not in excluded]
if not grains:
    raise ValueError("There are no specimens to export.")
for grain in grains:
    grain.measure()

metadata = dict(image_path=str(image_path), image_id=image_path.stem, sample_id=sample_id,
                species=species, depth_ccsf=depth_ccsf, Age_Ma=age_ma)
summary = sef.get_summary(grains, pixels_per_meter)
foram_data = sef.get_foram_morphometrics(summary, metadata=metadata)
result_dir = Path(mkdtemp(prefix="sef_results_", dir="/content"))
foram_data_path = result_dir / "foram_data.csv"
foram_data.to_csv(foram_data_path, index=False)
if export_crops_and_masks:
    export_dir = export_specimens(
        image, grains, foram_data, metadata, foram_data_path,
        output_dir=result_dir / "specimens",
    )
    # Use portable references within the downloadable archive.
    import pandas as pd
    manifest_path = export_dir / "manifest.csv"
    manifest = pd.read_csv(manifest_path)
    manifest["source_image"] = image_path.name
    manifest["morphometrics_csv"] = foram_data_path.relative_to(result_dir).as_posix()
    manifest.to_csv(manifest_path, index=False)
if editor is not None:
    editor.deactivate()
if export_crops_and_masks:
    print(f"Exported measurements, crops, and masks for {len(foram_data)} specimens.")
else:
    print(f"Exported measurements for {len(foram_data)} specimens.")
display(foram_data.head())

### Download results

With crops and masks, this downloads a ZIP containing the measurements, individual crops, binary masks, and manifest. Without them, it downloads the measurements CSV. Keep the source microscope image separately.

In [ ]:
from shutil import copyfile, make_archive

if export_crops_and_masks:
    download_path = make_archive(str(result_dir), "zip", root_dir=result_dir)
else:
    download_path = foram_data_path.with_name(f"{sample_id}_foram_data.csv")
    copyfile(foram_data_path, download_path)
files.download(str(download_path))